####Requirement
1. Read raw data from flight_time_raw table
2. Apply transformations to time values as hour to minute interval

    1. CRS_DEP_TIME
    2. DEP_TIME
    3. WHEELS_ON
    4. CRS_ARR_TIME
    5. ARR_TIME
3. Apply transformation to TAXI_IN to make it a minute interval

In [0]:
%sql

select * from dev.spark_db.flight_time_raw

####1. Read data to create a dataframe

In [0]:
flight_time_raw_df = spark.read.table("dev.spark_db.flight_time_raw")

####2. Develop logic to transform CRS_DEP_TIME to an interval

In [0]:
from pyspark.sql.functions import expr

step_1_df = (
    flight_time_raw_df.withColumns({
    "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    })
)

step_2_df = (
    step_1_df.withColumns({
        "CRS_DEP_TIME_NEW": expr("cast(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)")
    })
)

In [0]:
step_2_df.limit(2).display()

####3. Develop a reusable function

In [0]:
 def get_interval(hhmm_value):
     from pyspark.sql.functions import expr

     return expr(f"""
                 cast(concat(left(lpad({hhmm_value}, 4, '0'), 2), ':', 
                             right(lpad({hhmm_value}, 4, '0'), 2)) 
                             AS INTERVAL HOUR TO MINUTE)
                 """)

####4. Apply function to dataframe

In [0]:
result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME": get_interval("CRS_DEP_TIME"),
        "DEP_TIME": get_interval("DEP_TIME"),
        "WHEELS_ON": get_interval("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval("CRS_ARR_TIME"),
        "ARR_TIME": get_interval("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

####5. Save results to the table 

In [0]:
result_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time")

In [0]:
%sql

select * from dev.spark_db.flight_time